# Conheça o Colab

In [5]:
# -*- coding: utf-8 -*-
"""
Atividade: Processar PDFs da Aula 2 com Docling
Conversão de 3 PDFs para Markdown
"""

import os
import subprocess
import sys
import zipfile
import requests
from pathlib import Path
from google.colab import files
from docling.document_converter import DocumentConverter
import logging

# Configurar logging para ver o progresso
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==================== CONFIGURAÇÃO ====================
PASTA_AULA = "aula_2"
URLS_PDFS = {
    "bioetica_e_ia.pdf": "https://drive.google.com/uc?export=download&id=SEU_ID_1",
    "escrita_academica_ia.pdf": "https://drive.google.com/uc?export=download&id=SEU_ID_2",
    "twitter_algoritmo.pdf": "https://drive.google.com/uc?export=download&id=SEU_ID_3"
}

# ==================== FUNÇÕES AUXILIARES ====================

def criar_pasta_aula():
    """Cria a pasta aula_2 se não existir"""
    if not os.path.exists(PASTA_AULA):
        os.makedirs(PASTA_AULA)
        logger.info(f" Pasta '{PASTA_AULA}' criada com sucesso!")
    else:
        logger.info(f" Pasta '{PASTA_AULA}' já existe.")

def baixar_pdfs_do_drive():
    """
    Faz o upload dos PDFs manualmente (via Colab) para garantir que estão no ambiente
    Esta função substitui o download direto do Drive que pode ter problemas de autenticação
    """
    print("\n" + "="*60)
    print(" FAÇA O UPLOAD DOS 3 PDFs")
    print("="*60)
    print("Por favor, faça o upload dos seguintes arquivos:")
    print("  1. bioetica_e_ia.pdf")
    print("  2. escrita_academica_ia.pdf")
    print("  3. twitter_algoritmo.pdf")
    print("="*60 + "\n")

    arquivos_baixados = []

    for i in range(1, 4):
        print(f"\n Documento {i}/3 - Clique em 'Escolher arquivo' e selecione o PDF")
        uploaded = files.upload()

        for nome_arquivo, conteudo in uploaded.items():
            # Salvar o arquivo na pasta aula_2
            caminho_destino = os.path.join(PASTA_AULA, nome_arquivo)
            with open(caminho_destino, 'wb') as f:
                f.write(conteudo)
            arquivos_baixados.append(caminho_destino)
            logger.info(f" Arquivo salvo: {caminho_destino}")

    return arquivos_baixados

def verificar_pdfs_baixados():
    """Verifica quais PDFs estão disponíveis na pasta aula_2"""
    pdfs_encontrados = []
    for arquivo in os.listdir(PASTA_AULA):
        if arquivo.endswith('.pdf'):
            caminho = os.path.join(PASTA_AULA, arquivo)
            pdfs_encontrados.append(caminho)
            logger.info(f"📄 PDF encontrado: {arquivo} ({os.path.getsize(caminho)} bytes)")
    return pdfs_encontrados

# ==================== CONVERSÃO PARA MARKDOWN ====================

def converter_pdf_para_markdown(caminho_pdf):
    """
    Converte um único PDF para Markdown usando Docling
    Retorna o caminho do arquivo .md gerado
    """
    try:
        nome_base = os.path.splitext(os.path.basename(caminho_pdf))[0]
        caminho_md = os.path.join(PASTA_AULA, f"{nome_base}.md")

        logger.info(f" Convertendo: {nome_base}.pdf")

        # Inicializar o conversor do Docling
        converter = DocumentConverter()

        # Realizar a conversão
        resultado = converter.convert(caminho_pdf)

        # Extrair o conteúdo em Markdown
        markdown_texto = resultado.document.export_to_markdown()

        # Salvar o arquivo Markdown
        with open(caminho_md, 'w', encoding='utf-8') as f:
            f.write(markdown_texto)

        logger.info(f" Convertido: {nome_base}.md")
        return caminho_md

    except Exception as e:
        logger.error(f" Erro ao converter {caminho_pdf}: {str(e)}")
        return None

def converter_todos_pdfs():
    """Converte todos os PDFs da pasta aula_2 para Markdown"""
    pdfs = verificar_pdfs_baixados()

    if not pdfs:
        logger.warning(" Nenhum PDF encontrado na pasta 'aula_2'.")
        return []

    logger.info(f"\n Iniciando conversão de {len(pdfs)} PDFs...")
    arquivos_md = []

    for pdf in pdfs:
        md = converter_pdf_para_markdown(pdf)
        if md:
            arquivos_md.append(md)

    return arquivos_md

# ==================== RELATÓRIO E FINALIZAÇÃO ====================

def gerar_relatorio(arquivos_md):
    """Gera um relatório com informações sobre os arquivos convertidos"""
    if not arquivos_md:
        print("\n Nenhum arquivo foi convertido para Markdown.")
        return

    print("\n" + "="*60)
    print(" RELATÓRIO DE CONVERSÃO")
    print("="*60)

    for i, arquivo in enumerate(arquivos_md, 1):
        if os.path.exists(arquivo):
            with open(arquivo, 'r', encoding='utf-8') as f:
                conteudo = f.read()
                palavras = len(conteudo.split())
                caracteres = len(conteudo)

                print(f"\n Documento {i}: {os.path.basename(arquivo)}")
                print(f"   Palavras: {palavras:,}")
                print(f"    Caracteres: {caracteres:,}")
                print(f"   Tamanho: {os.path.getsize(arquivo):,} bytes")

    print("\n" + "="*60)
    print(f" Total de arquivos convertidos: {len(arquivos_md)}")
    print("="*60)

def baixar_resultados():
    """Cria um ZIP com os arquivos Markdown e disponibiliza para download"""
    arquivos_md = [f for f in os.listdir(PASTA_AULA) if f.endswith('.md')]

    if not arquivos_md:
        print(" Nenhum arquivo Markdown encontrado para baixar.")
        return

    nome_zip = "aula_2_markdowns.zip"
    caminho_zip = os.path.join(PASTA_AULA, nome_zip)

    with zipfile.ZipFile(caminho_zip, 'w') as zipf:
        for arquivo in arquivos_md:
            caminho_completo = os.path.join(PASTA_AULA, arquivo)
            zipf.write(caminho_completo, arquivo)
            print(f" Adicionado ao ZIP: {arquivo}")

    print(f"\n Baixando o arquivo ZIP: {nome_zip}")
    files.download(caminho_zip)

# ==================== FUNÇÃO PRINCIPAL ====================

def main():
    """Função principal que executa toda a atividade"""
    print(" ATIVIDADE: PROCESSAR PDFS DA AULA 2 COM DOCLING")
    print("="*60)

    try:
        # 1. Criar a pasta aula_2
        criar_pasta_aula()

        # 2. Baixar os PDFs (via upload manual no Colab)
        print("\n Etapa 1: Upload dos PDFs")
        print("-" * 40)
        pdfs_baixados = baixar_pdfs_do_drive()

        if not pdfs_baixados:
            print("❌ Nenhum PDF foi enviado. Encerrando...")
            return

        # 3. Verificar os PDFs na pasta
        print("\n Etapa 2: Verificando PDFs")
        print("-" * 40)
        pdfs = verificar_pdfs_baixados()

        if len(pdfs) != 3:
            print(f" Atenção: Foram encontrados {len(pdfs)} PDFs, mas esperava 3.")
            print("   Verifique se todos os arquivos foram enviados corretamente.")

        # 4. Converter para Markdown
        print("\n Etapa 3: Convertendo PDFs para Markdown")
        print("-" * 40)
        arquivos_md = converter_todos_pdfs()

        # 5. Gerar relatório
        gerar_relatorio(arquivos_md)

        # 6. Baixar os resultados
        print("\n Etapa 4: Download dos resultados")
        print("-" * 40)
        baixar_resultados()

        print("\n" + "="*60)
        print(" ATIVIDADE CONCLUÍDA COM SUCESSO!")
        print("="*60)

    except Exception as e:
        print(f"\n ERRO: {str(e)}")
        print("Por favor, verifique os arquivos e tente novamente.")

# ==================== EXECUÇÃO ====================
if __name__ == "__main__":
    main()

 ATIVIDADE: PROCESSAR PDFS DA AULA 2 COM DOCLING

 Etapa 1: Upload dos PDFs
----------------------------------------

 FAÇA O UPLOAD DOS 3 PDFs
Por favor, faça o upload dos seguintes arquivos:
  1. bioetica_e_ia.pdf
  2. escrita_academica_ia.pdf
  3. twitter_algoritmo.pdf


 Documento 1/3 - Clique em 'Escolher arquivo' e selecione o PDF


Saving bioetica_e_ia (1).pdf to bioetica_e_ia (1).pdf

 Documento 2/3 - Clique em 'Escolher arquivo' e selecione o PDF


Saving twitter_algoritmo.pdf to twitter_algoritmo.pdf

 Documento 3/3 - Clique em 'Escolher arquivo' e selecione o PDF


[INFO] 2026-08-05 20:00:38,452 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-08-05 20:00:38,455 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-08-05 20:00:38,473 [RapidOCR] download_file.py:60: File exists and is valid: /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth
[INFO] 2026-08-05 20:00:38,475 [RapidOCR] main.py:50: Using /usr/local/lib/python3.12/dist-packages/rapidocr/models/PP-OCRv6_det_small.pth


Saving escrita_academica_ia.pdf to escrita_academica_ia.pdf

 Etapa 2: Verificando PDFs
----------------------------------------
 Atenção: Foram encontrados 6 PDFs, mas esperava 3.
   Verifique se todos os arquivos foram enviados corretamente.

 Etapa 3: Convertendo PDFs para Markdown
----------------------------------------


...

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]


 RELATÓRIO DE CONVERSÃO

 Documento 1: twitter_algoritmo (2).md
   Palavras: 7,828
    Caracteres: 54,440
   Tamanho: 56,135 bytes

 Documento 2: bioetica_e_ia (2).md
   Palavras: 7,354
    Caracteres: 51,213
   Tamanho: 52,745 bytes

 Documento 3: escrita_academica_ia.md
   Palavras: 6,108
    Caracteres: 42,678
   Tamanho: 43,754 bytes

 Documento 4: escrita_academica_ia (2).md
   Palavras: 6,108
    Caracteres: 42,678
   Tamanho: 43,754 bytes

 Documento 5: twitter_algoritmo.md
   Palavras: 7,828
    Caracteres: 54,440
   Tamanho: 56,135 bytes

 Documento 6: bioetica_e_ia (1).md
   Palavras: 7,354
    Caracteres: 51,213
   Tamanho: 52,745 bytes

 Total de arquivos convertidos: 6

 Etapa 4: Download dos resultados
----------------------------------------
 Adicionado ao ZIP: bioetica_e_ia (1).md
 Adicionado ao ZIP: escrita_academica_ia.md
 Adicionado ao ZIP: bioetica_e_ia (2).md
 Adicionado ao ZIP: escrita_academica_ia (2).md
 Adicionado ao ZIP: twitter_algoritmo (2).md
 Adicionado a

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


 ATIVIDADE CONCLUÍDA COM SUCESSO!


In [9]:
# -*- coding: utf-8 -*-
"""
Tarefa 2 - Extração de metadados com Structured Outputs (OpenRouter)
Versão com validação flexível de campos
"""

import os
import json
import requests
from pathlib import Path
from typing import Dict, List, Optional, Any
import time
from datetime import datetime

# ==================== ACESSAR COLAB SECRETS ====================
from google.colab import userdata

try:
    OPENROUTER_API_KEY = userdata.get('aula02')
    print(" Chave da API carregada do Colab Secrets")
except Exception as e:
    print(f" Erro ao acessar o Colab Secrets: {e}")
    OPENROUTER_API_KEY = None

# ==================== CONFIGURAÇÃO ====================
PASTA_AULA = "aula_2"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
MODELO = "openai/gpt-4o"

# ==================== FUNÇÃO PARA NORMALIZAR CAMPOS ====================

def normalizar_metadados(dados: Dict) -> Dict:
    """
    Converte campos com nomes em português para o padrão esperado.
    """
    # Mapeamento de campos possíveis para o padrão
    mapeamento = {
        'título': 'titulo',
        'titulo': 'titulo',
        'Título': 'titulo',
        'autores': 'autores',
        'Autores': 'autores',
        'ano': 'ano',
        'Ano': 'ano',
        'Ano de publicação': 'ano',
        'ano de publicação': 'ano'
    }

    dados_normalizados = {}

    for chave, valor in dados.items():
        chave_normalizada = mapeamento.get(chave, chave.lower())
        dados_normalizados[chave_normalizada] = valor

    return dados_normalizados

# ==================== FUNÇÃO PRINCIPAL DE EXTRAÇÃO ====================

def extrair_metadados_com_structured_outputs(conteudo_md: str, nome_arquivo: str = "") -> Optional[Dict]:
    """
    Extrai metadados usando OpenRouter com JSON Output.
    """

    # ==================== PREPARAR O PROMPT ====================
    prompt = f"""
    Extraia os seguintes metadados do documento acadêmico em Markdown:

    Campos a extrair (use EXATAMENTE estes nomes):
    - titulo: Título do trabalho
    - autores: Lista de autores (array de strings)
    - ano: Ano de publicação (número inteiro)

    Documento:
    {conteudo_md[:8000]}

    IMPORTANTE: Responda APENAS com um JSON válido contendo os campos 'titulo', 'autores' e 'ano'.
    Exemplo de resposta esperada:
    {{"titulo": "Título do Trabalho", "autores": ["Autor 1", "Autor 2"], "ano": 2024}}
    """

    # ==================== CONFIGURAR A REQUISIÇÃO ====================
    headers = {
        "Authorization": f"Bearer {OPENROUTER_API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": "https://colab.research.google.com/",
        "X-Title": "Extracao Metadados"
    }

    payload = {
        "model": MODELO,
        "messages": [
            {
                "role": "system",
                "content": "Você é um assistente que extrai metadados de documentos acadêmicos. Responda SEMPRE em JSON com os campos 'titulo', 'autores' e 'ano'."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "response_format": {"type": "json_object"},
        "temperature": 0.1,
        "max_tokens": 500
    }

    # ==================== EXECUTAR A REQUISIÇÃO ====================
    try:
        print(f" Enviando requisição para OpenRouter (modelo: {MODELO})...")
        if nome_arquivo:
            print(f"    Processando: {nome_arquivo}")

        inicio = time.time()
        response = requests.post(
            OPENROUTER_URL,
            headers=headers,
            json=payload,
            timeout=90
        )
        tempo = time.time() - inicio

        if response.status_code != 200:
            print(f" Status Code: {response.status_code}")
            print(f"   Resposta: {response.text[:300]}")
            return None

        result = response.json()

        if "choices" not in result or not result["choices"]:
            print(" Resposta sem 'choices'")
            return None

        conteudo_resposta = result["choices"][0]["message"]["content"]

        # Tentar extrair JSON da resposta
        try:
            conteudo_limpo = conteudo_resposta.strip()
            if conteudo_limpo.startswith('```json'):
                conteudo_limpo = conteudo_limpo[7:]
            if conteudo_limpo.startswith('```'):
                conteudo_limpo = conteudo_limpo[3:]
            if conteudo_limpo.endswith('```'):
                conteudo_limpo = conteudo_limpo[:-3]
            conteudo_limpo = conteudo_limpo.strip()

            dados = json.loads(conteudo_limpo)

            # Normalizar os campos (português -> inglês)
            dados_normalizados = normalizar_metadados(dados)

        except json.JSONDecodeError as e:
            print(f" Erro ao decodificar JSON: {e}")
            print(f"   Conteúdo: {conteudo_resposta[:200]}...")
            return None

        # Validar estrutura
        if not all(k in dados_normalizados for k in ["titulo", "autores", "ano"]):
            print(" Resposta não contém todos os campos obrigatórios")
            print(f"   Campos recebidos: {list(dados_normalizados.keys())}")
            # Tentar inferir campos faltantes
            if 'titulo' not in dados_normalizados and 'Título' in dados:
                dados_normalizados['titulo'] = dados['Título']
            if 'autores' not in dados_normalizados and 'Autores' in dados:
                dados_normalizados['autores'] = dados['Autores']
            if 'ano' not in dados_normalizados and 'Ano' in dados:
                dados_normalizados['ano'] = dados['Ano']

            if not all(k in dados_normalizados for k in ["titulo", "autores", "ano"]):
                return None

        # Garantir que autores seja uma lista
        if not isinstance(dados_normalizados["autores"], list):
            if isinstance(dados_normalizados["autores"], str):
                dados_normalizados["autores"] = [dados_normalizados["autores"]]
            else:
                dados_normalizados["autores"] = []

        # Garantir que ano seja inteiro
        if not isinstance(dados_normalizados["ano"], int):
            try:
                dados_normalizados["ano"] = int(dados_normalizados["ano"])
            except:
                dados_normalizados["ano"] = None

        print(f" Extração concluída em {tempo:.2f}s")
        return dados_normalizados

    except requests.exceptions.Timeout:
        print(f" Timeout após 90 segundos")
        return None
    except requests.exceptions.RequestException as e:
        print(f" Erro na requisição: {e}")
        return None
    except Exception as e:
        print(f" Erro inesperado: {e}")
        return None

# ==================== FUNÇÕES DE PROCESSAMENTO ====================

def processar_todos_arquivos_markdown() -> Dict[str, Dict]:
    """
    Processa todos os arquivos .md na pasta aula_2.
    """
    if not OPENROUTER_API_KEY:
        print(" ERRO: Chave da API não encontrada!")
        return {}

    if not os.path.exists(PASTA_AULA):
        print(f" Pasta '{PASTA_AULA}' não encontrada!")
        return {}

    arquivos_md = [f for f in os.listdir(PASTA_AULA) if f.endswith('.md')]

    if not arquivos_md:
        print(f" Nenhum arquivo .md encontrado em '{PASTA_AULA}'")
        return {}

    print("\n" + "="*60)
    print(f" INICIANDO EXTRAÇÃO DE METADADOS")
    print(f"   Total de arquivos: {len(arquivos_md)}")
    print(f"   Modelo: {MODELO}")
    print("="*60 + "\n")

    resultados = {}
    erros = []

    for i, arquivo in enumerate(arquivos_md, 1):
        print(f"\n [{i}/{len(arquivos_md)}] Processando: {arquivo}")
        print("-" * 40)

        caminho_arquivo = os.path.join(PASTA_AULA, arquivo)

        try:
            with open(caminho_arquivo, 'r', encoding='utf-8') as f:
                conteudo = f.read()

            if not conteudo.strip():
                print(" Arquivo vazio!")
                resultados[arquivo] = {"erro": "Arquivo vazio"}
                continue

            metadados = extrair_metadados_com_structured_outputs(conteudo, arquivo)

            if metadados:
                resultados[arquivo] = metadados
                print(f"\n Metadados extraídos:")
                print(f"    Título: {metadados.get('titulo', 'N/A')[:60]}")
                print(f"    Autores: {', '.join(metadados.get('autores', ['N/A']))}")
                print(f"    Ano: {metadados.get('ano', 'N/A')}")
            else:
                print(f" Falha na extração")
                erros.append(arquivo)
                resultados[arquivo] = {"erro": "Falha na extração"}

            if i < len(arquivos_md):
                time.sleep(2)

        except Exception as e:
            print(f" Erro ao processar {arquivo}: {str(e)}")
            erros.append(arquivo)
            resultados[arquivo] = {"erro": str(e)}

    print("\n" + "="*60)
    print(" RESUMO DO PROCESSAMENTO")
    print("="*60)
    print(f" Sucessos: {len(resultados) - len(erros)}")
    print(f" Falhas: {len(erros)}")
    if erros:
        print(f"   Arquivos com erro: {', '.join(erros)}")

    return resultados

# ==================== FUNÇÃO PARA SALVAR RESULTADOS ====================

def salvar_resultados_estruturados(resultados: Dict[str, Dict]) -> str:
    if not resultados:
        print(" Nenhum resultado para salvar")
        return ""

    dados_finais = {
        "metadados_extracao": {
            "data_extracao": datetime.now().isoformat(),
            "modelo_utilizado": MODELO,
            "total_documentos": len(resultados),
            "documentos_processados": list(resultados.keys())
        },
        "documentos": []
    }

    for nome_arquivo, metadados in resultados.items():
        documento = {
            "arquivo": nome_arquivo,
            "metadados": metadados
        }
        dados_finais["documentos"].append(documento)

    caminho_json = os.path.join(PASTA_AULA, "metadados_extraidos.json")
    with open(caminho_json, 'w', encoding='utf-8') as f:
        json.dump(dados_finais, f, ensure_ascii=False, indent=2)

    print(f"\n Metadados salvos em: {caminho_json}")

    caminho_simples = os.path.join(PASTA_AULA, "metadados_simplificados.json")
    with open(caminho_simples, 'w', encoding='utf-8') as f:
        json.dump(resultados, f, ensure_ascii=False, indent=2)

    print(f" Versão simplificada em: {caminho_simples}")

    return caminho_json

# ==================== FUNÇÃO PARA EXIBIR RELATÓRIO ====================

def exibir_relatorio_completo(resultados: Dict[str, Dict]):
    if not resultados:
        return

    print("\n" + "="*60)
    print(" RELATÓRIO COMPLETO DOS METADADOS")
    print("="*60)

    for i, (arquivo, metadados) in enumerate(resultados.items(), 1):
        print(f"\n{i}. {arquivo}")
        print("-" * 40)

        if "erro" in metadados:
            print(f"    Erro: {metadados['erro']}")
            continue

        print(f"    Título: {metadados.get('titulo', 'N/A')}")
        print(f"    Autores: {', '.join(metadados.get('autores', ['N/A']))}")
        print(f"    Ano: {metadados.get('ano', 'N/A')}")

# ==================== FUNÇÃO PRINCIPAL ====================

def main():
    print("="*60)
    print(" TAREFA 2 - EXTRAÇÃO DE METADADOS")
    print("   Usando OpenRouter com JSON Output")
    print("="*60)

    resultados = processar_todos_arquivos_markdown()

    if resultados:
        salvar_resultados_estruturados(resultados)
        exibir_relatorio_completo(resultados)

        print("\n" + "="*60)
        print(" TAREFA 2 CONCLUÍDA COM SUCESSO!")
        print(f"    {len(resultados)} documentos processados")
        print("="*60)
    else:
        print("\n Nenhum resultado foi gerado.")
        print("   Verifique:")
        print("   1. Sua chave da OpenRouter no Colab Secrets")
        print("   2. Os arquivos .md na pasta 'aula_2'")
        print("   3. Sua conexão com a internet")

# ==================== EXECUÇÃO ====================
if __name__ == "__main__":
    main()

 Chave da API carregada do Colab Secrets
 TAREFA 2 - EXTRAÇÃO DE METADADOS
   Usando OpenRouter com JSON Output

 INICIANDO EXTRAÇÃO DE METADADOS
   Total de arquivos: 6
   Modelo: openai/gpt-4o


 [1/6] Processando: bioetica_e_ia (1).md
----------------------------------------
 Enviando requisição para OpenRouter (modelo: openai/gpt-4o)...
    Processando: bioetica_e_ia (1).md
 Extração concluída em 1.16s

 Metadados extraídos:
    Título: Entre o algoritmo e o Juramento de Hipócrates: bioética na e
    Autores: Juracy Barbosa dos Santos, Guilhermina Rego, Rui Nunes
    Ano: 2023

 [2/6] Processando: escrita_academica_ia.md
----------------------------------------
 Enviando requisição para OpenRouter (modelo: openai/gpt-4o)...
    Processando: escrita_academica_ia.md
 Extração concluída em 1.49s

 Metadados extraídos:
    Título: Escrita acadêmica ética, responsável e humana com inteligênc
    Autores: Rafael Cardoso Sampaio
    Ano: 2025

 [3/6] Processando: bioetica_e_ia (2).md
----